# Notebook 3: ESM-2 Classifier Training & Evaluation

This notebook trains and evaluates the multi-label tropism classifier on ESM-2 capsid embeddings.

**Labels:** CNS tropic | Peripheral tropic | Broad tropic | BBB-crossing  
**Architecture:** MLP (3 hidden layers, dropout)  
**Training:** 5-fold CV + threshold optimization  
**Baseline comparison:** Logistic Regression, Random Forest on physicochemical features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Project imports
import sys
sys.path.insert(0, '..')
from src.models.classifier import (
    load_labels, cross_validate, train_mlp, predict_mlp,
    get_baseline_models, LABEL_COLS, save_model
)
from src.visualization.plots import plot_roc_curves, plot_embedding_umap

# Config
EMBEDDINGS_PATH = '../data/processed/embeddings.npz'
LABELS_PATH = '../data/raw/labels.csv'
MODEL_OUTPUT = '../models/'
FIGURES_OUTPUT = '../results/figures/'

print('Setup complete')

## 1. Load Data

In [ ]:
# Load ESM-2 embeddings
data = np.load(EMBEDDINGS_PATH, allow_pickle=True)
embeddings = data['embeddings']
names = list(data['names'])

print(f'Embeddings shape: {embeddings.shape}')
print(f'Number of capsids: {len(names)}')
print(f'Embedding dimension: {embeddings.shape[1]}')

# Load labels
y, valid_names = load_labels(LABELS_PATH, names)
valid_idx = [names.index(n) for n in valid_names]
X = embeddings[valid_idx]

print(f'\nLabeled sequences: {len(valid_names)}')
print('\nLabel distribution:')
df_labels = pd.DataFrame(y, columns=LABEL_COLS, index=valid_names)
print(df_labels.sum().astype(int).to_string())

In [ ]:
# Visualize label co-occurrence
fig, ax = plt.subplots(figsize=(6, 5))
co_occur = df_labels.T @ df_labels
im = ax.imshow(co_occur, cmap='Blues')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(LABEL_COLS, rotation=45)
ax.set_yticklabels(LABEL_COLS)
for i in range(4):
    for j in range(4):
        ax.text(j, i, int(co_occur.iloc[i, j]), ha='center', va='center', fontweight='bold')
plt.colorbar(im)
ax.set_title('Label Co-occurrence Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. UMAP Visualization of Embedding Space

In [ ]:
plot_embedding_umap(
    X, valid_names,
    labels=y,
    label_names=LABEL_COLS,
    output_path=f'{FIGURES_OUTPUT}/umap_capsid_space.png',
    title='ESM-2 Embedding Space — AAV Capsids'
)

from IPython.display import Image
Image(f'{FIGURES_OUTPUT}/umap_capsid_space.png', width=600)

## 3. Cross-Validation: MLP Classifier

In [ ]:
print('Running 5-fold cross-validation (MLP on ESM-2 embeddings)...')
print('This takes ~5 minutes on CPU\n')

cv_results = cross_validate(X, y, valid_names, n_folds=5, use_mlp=True)

## 4. Baseline Comparison

In [ ]:
# Load physicochemical features for baseline comparison
try:
    physchem_data = np.load('../data/processed/embeddings_physicochemical.npz', allow_pickle=True)
    X_physchem = physchem_data['embeddings']
    physchem_names = list(physchem_data['names'])
    y_physchem, _ = load_labels(LABELS_PATH, physchem_names)
    
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold
    from sklearn.multioutput import MultiOutputClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score
    
    print('Baseline: Random Forest on physicochemical features')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    baseline_aurocs = {label: [] for label in LABEL_COLS}
    
    for train_idx, val_idx in cv.split(X_physchem, y_physchem[:, 0]):
        X_tr, X_val = X_physchem[train_idx], X_physchem[val_idx]
        y_tr, y_val = y_physchem[train_idx], y_physchem[val_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_val_s = scaler.transform(X_val)
        clf = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
        clf.fit(X_tr_s, y_tr)
        for i, label in enumerate(LABEL_COLS):
            if len(np.unique(y_val[:, i])) < 2:
                continue
            prob = clf.estimators_[i].predict_proba(X_val_s)[:, 1]
            baseline_aurocs[label].append(roc_auc_score(y_val[:, i], prob))
    
    print('\nBaseline AUROC (physicochemical features + RF):')
    for label in LABEL_COLS:
        mean = np.mean(baseline_aurocs[label]) if baseline_aurocs[label] else float('nan')
        print(f'  {label:<15} {mean:.3f}')
except FileNotFoundError:
    print('[INFO] Physicochemical baseline not computed yet.')
    print('Run: python src/features/esm_embeddings.py --method physicochemical')

## 5. Train Final Model

In [ ]:
print('Training final model on all labeled data...')
final_model, scaler = train_mlp(X, y, input_dim=X.shape[1], epochs=150)
save_model(final_model, scaler, MODEL_OUTPUT)
print('Model saved!')

## 6. Predictions on Known Serotypes

In [ ]:
probs = predict_mlp(final_model, scaler, X)

results_df = pd.DataFrame(probs, columns=LABEL_COLS, index=valid_names)
results_df['top_label'] = results_df.idxmax(axis=1)

print('=== Predicted Tropism Scores for All Capsids ===')
print(results_df.round(3).to_string())

# Save
import os
os.makedirs('../results/predictions', exist_ok=True)
results_df.to_csv('../results/predictions/all_serotypes.csv')

# Highlight known CNS/BBB capsids
print('\n=== Key CNS/BBB Capsids (Known Phenotype Check) ===')
cns_bbb = ['AAV9', 'PHP.eB', 'PHP.B', 'AAV-BR1']
for capsid in cns_bbb:
    if capsid in results_df.index:
        row = results_df.loc[capsid]
        print(f'{capsid:<12}: CNS={row["cns"]:.3f}  BBB={row["bbb"]:.3f}  [Expected: high]')

In [ ]:
from src.visualization.plots import plot_serotype_predictions
plot_serotype_predictions(
    '../results/predictions/all_serotypes.csv',
    f'{FIGURES_OUTPUT}/serotype_predictions.png'
)
Image(f'{FIGURES_OUTPUT}/serotype_predictions.png', width=750)